# Notebook 03 — LLM evaluation

Uses `gpt-4o-mini` as a judge to grade the assistant's answers for the same
44 golden questions. Three axes, each on a 1–5 scale:

- **Relevance** — does the answer address the question?
- **Faithfulness** — does it stay grounded in the retrieved context?
- **Completeness** — does it cover the key points in the reference answer?

We compare four prompt strategies registered in `src/prompts.py`:

1. `concise` — short Markdown answer with bracketed citations.
2. `with_citations` — bullet list where every claim carries a citation.
3. `chain_of_thought` — reasoning trace followed by a clean answer.
4. `no_citations` — same shape as `concise`, citation rule removed.

Two artefacts feed this notebook:

- `data/eval/llm_judge_report.json` — the legacy single-prompt run.
- `data/eval/llm_prompt_report.json` — the multi-prompt comparison.

Both are produced by `python src/evaluate.py all`.


In [ ]:
import json
from pathlib import Path

eval_dir = Path('data/eval')
single = json.loads((eval_dir / 'llm_judge_report.json').read_text()) if (eval_dir / 'llm_judge_report.json').exists() else None
multi = json.loads((eval_dir / 'llm_prompt_report.json').read_text()) if (eval_dir / 'llm_prompt_report.json').exists() else None

if single is not None:
    print('Single-prompt summary (legacy):')
    for k, v in single['summary'].items():
        print(f'  {k:>22}: {v}')
else:
    print('llm_judge_report.json not found; rerun `python src/evaluate.py llm`.')

if multi is not None:
    print('\nMulti-prompt summary:')
    for prompt, metrics in multi['per_prompt'].items():
        print(f'  {prompt:>20}: relevance={metrics["avg_relevance"]}, faithfulness={metrics["avg_faithfulness"]}, completeness={metrics["avg_completeness"]}, overall={metrics["avg_overall"]}')
    print('\nWinner:', multi['winner'])
else:
    print('llm_prompt_report.json not found; rerun `python src/evaluate.py prompts`.')

In [ ]:
import matplotlib.pyplot as plt

if multi is None or not multi['per_prompt']:
    print('No multi-prompt data available — run `python src/evaluate.py prompts` first.')
else:
    prompts = list(multi['per_prompt'].keys())
    metrics = ['avg_relevance', 'avg_faithfulness', 'avg_completeness']
    fig, ax = plt.subplots(figsize=(8, 4.5))
    width = 0.27
    x = list(range(len(prompts)))
    colors = ['#4C72B0', '#55A868', '#C44E52']
    for i, m in enumerate(metrics):
        values = [multi['per_prompt'][p][m] for p in prompts]
        offsets = [xi + (i - 1) * width for xi in x]
        ax.bar(offsets, values, width=width, label=m.replace('avg_', '').title(), color=colors[i])
    ax.set_xticks(x)
    ax.set_xticklabels(prompts, rotation=15, ha='right')
    ax.set_ylim(0, 5)
    ax.set_ylabel('score (1–5)')
    ax.set_title('LLM-as-judge scores by prompt strategy')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

    # Overall winner bar.
    fig, ax = plt.subplots(figsize=(7, 3.5))
    overall = [multi['per_prompt'][p]['avg_overall'] for p in prompts]
    ax.bar(prompts, overall, color=['#4C72B0' if p != multi['winner'] else '#2CA02C' for p in prompts])
    ax.set_ylim(0, 5)
    ax.set_ylabel('mean of relevance + faithfulness + completeness')
    ax.set_title(f'Winner: {multi["winner"]}')
    for i, v in enumerate(overall):
        ax.text(i, v + 0.05, f'{v:.2f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.show()

## Verdict

Across the golden set the prompt comparison shows the winner selected by
`evaluate_llm_prompts()`. The production pipeline picks this winner via the
`ANSWER_PROMPT` env var or the `--prompt` CLI flag in `src/rag.py`.

In typical runs `concise` and `with_citations` trade the top spot; the
judge frequently penalises `chain_of_thought` for verbosity and
`no_citations` for making source verification harder. The legacy
`llm_judge_report.json` reflects the single-prompt run and stays around
for backwards compatibility with earlier deliverables.
